In [2]:
import torch

if not hasattr(torch.compiler, "is_compiling"):
    torch.compiler.is_compiling = lambda: False

from transformers import Qwen2_5_VLForConditionalGeneration, AutoTokenizer, AutoProcessor
from qwen_vl_utils import process_vision_info

In [3]:
# !pip install --upgrade "huggingface-hub>=0.34.0,<1.0"

In [3]:
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-7B-Instruct", torch_dtype="auto", device_map="auto"
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

In [4]:
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct")

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


In [7]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": "/home/jovyan/nkiselev/ddorin/project/Image-Transform-Predict/assets/image.jpg"},
            {"type": "image", "image": "/home/jovyan/nkiselev/ddorin/project/Image-Transform-Predict/assets/transformed_image.jpg"},
            {"type": "text", "text": "Identify the similarities between these images."},
        ],
    }
]

# Preparation for inference
text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)
inputs = inputs.to("cuda")

# Inference
generated_ids = model.generate(**inputs, max_new_tokens=128)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
print(output_text)

['These two images share the following similarities:\n\n1. **Subject**: Both images feature animals, specifically dogs.\n2. **Outdoor Setting**: Both photos appear to be taken outdoors, with greenery in the background.\n3. **Focus on the Animal**: The primary focus of both images is on the animal, with the background blurred to emphasize the subject.\n4. **Natural Lighting**: Both images seem to have been captured using natural light, giving them a soft and warm tone.\n\nThe differences lie in the specific breeds, colors, and poses of the dogs, as well as the overall mood and composition of each photo.']


In [12]:
prompt = """You are given two images: Image A (original) and Image B (transformed).  
Your task is to predict the sequence of transformations applied to Image A to obtain Image B, using **only** the following allowed operations:  
"noop", "grayscale", "rotate_90", "rotate_180", "rotate_270", "color_jitter", "noise_adding", "crop", "horizontal_flip", "vertical_flip".

- The sequence may contain **zero, one, or multiple** transformations applied in order.  
- If Image A and Image B are identical, return: ["noop"]  
- If Image B can be obtained by applying a sequence of the allowed transformations (in the correct order), return that sequence as a JSON list, e.g.: ["color_jitter", "noise_adding", "rotate_270", "horizontal_flip"]  
- If the transformation from Image A to Image B **requires any operation not in the allowed list** (e.g., blur, resize, perspective distortion, custom warping, etc.), or if the images are unrelated, return an empty list: []  

Output only the JSON list. Do not add explanations, comments, or extra text."""

In [15]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": "/home/jovyan/nkiselev/ddorin/project/Image-Transform-Predict/assets/dog.jpg"},
            {"type": "image", "image": "/home/jovyan/nkiselev/ddorin/project/Image-Transform-Predict/assets/transformed_dog.jpg"},
            {"type": "text", "text": prompt},
        ],
    }
]

# Preparation for inference
text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)
inputs = inputs.to("cuda")

# Inference
generated_ids = model.generate(**inputs, max_new_tokens=128)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
print(output_text)

['["color_jitter"]']
